# 01_fit_msaa_flexible.ipynb

Flexible MS-AA fitting notebook.

In [16]:

ANALYSIS_TYPE = "spatial"   # "spatial" or "temporal"
FIT_SCOPE = "within"         # "within" or "across"

K_VALUES = [5, 10]
CONDITIONS_TO_USE = ["intact", "word", "rest"]
TRIM_REST_BY = 100
RNG_SEED = 42

PIEMAN_MAT_PATH = "data/pieman/raw/pieman_data.mat"
POSTERIOR_MAT_PATH = "data/pieman/raw/pieman_posterior_K700.mat"
HELPERS_DIR = "helpers"
HYPERTOOLS_NORMALIZE_PATH = "hypertools/normalize.py"

OUTPUT_DIR = "msaa_flexible_outputs_npz"

MSAA_OPTS = dict(maxiter=200, conv_crit=1e-6, fix_var_iter=5, heteroscedastic=False, use_gpu=False, rngSEED=RNG_SEED)
SKIP_EXISTING = True
STOP_ON_ERROR = False
VERBOSE_PROGRESS = True


In [17]:

%matplotlib inline
import os, sys, importlib.util
import numpy as np, pandas as pd
from scipy.io import loadmat

HT_NORMALIZE_AVAILABLE = False
try:
    spec = importlib.util.spec_from_file_location("normalize", HYPERTOOLS_NORMALIZE_PATH)
    ht = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ht)
    HT_NORMALIZE_AVAILABLE = True
except Exception:
    ht = None

sys.path.append(HELPERS_DIR)
from MultiSubject_AA import Subject, multi_subject_aa
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Ready.")


Ready.


In [18]:

def to_float_array(x): return np.array(x, dtype=float)

def array_cutter(data_list, cut_length):
    out = [to_float_array(x).copy() for x in data_list]
    return out if cut_length <= 0 else [x[:-cut_length] for x in out]

def coerce_TxV(x):
    x = to_float_array(x)
    if x.ndim != 2:
        raise ValueError(f"Expected 2D array, got {x.shape}")
    T, V = x.shape
    if V < T:
        x = x.T
    x = x - x.mean(axis=0, keepdims=True)
    return x.astype(np.float32, copy=False)

def normalize_across_fallback(subject_list, eps=1e-8):
    subj = [coerce_TxV(x) for x in subject_list]
    stacked = np.vstack(subj)
    mu = stacked.mean(axis=0, keepdims=True)
    sd = stacked.std(axis=0, keepdims=True) + eps
    return [((x - mu) / sd).astype(np.float32, copy=False) for x in subj]

def normalize_subject_list(subject_list):
    subj = [coerce_TxV(x) for x in subject_list]
    if HT_NORMALIZE_AVAILABLE:
        try:
            return ht.normalize(subj, normalize="across")
        except Exception:
            return normalize_across_fallback(subj)
    return normalize_across_fallback(subj)

def load_condition_data():
    pieman_data = loadmat(PIEMAN_MAT_PATH)
    data, conds = [], []
    for c in CONDITIONS_TO_USE:
        next_data = list(map(lambda i: pieman_data[c][:, i][0], np.arange(pieman_data[c].shape[1])))
        data.extend(next_data)
        conds.extend([c] * len(next_data))
    conds_array = np.array(conds)
    condition_data = {}
    for c in CONDITIONS_TO_USE:
        cur = [data[i] for i in np.where(conds_array == c)[0]]
        if c == "rest" and TRIM_REST_BY > 0:
            cur = array_cutter(cur, TRIM_REST_BY)
        condition_data[c] = normalize_subject_list(cur)
    return condition_data

condition_data = load_condition_data()
for c in CONDITIONS_TO_USE:
    print(c, len(condition_data[c]), np.asarray(condition_data[c][0]).shape)
posterior = loadmat(POSTERIOR_MAT_PATH)
centers = to_float_array(posterior['posterior']['centers'][0][0][0][0][0])
widths = to_float_array(list(posterior['posterior']['widths'][0][0][0][0][0][:, 0].T)).ravel()


intact 36 (300, 700)
word 36 (300, 700)
rest 36 (300, 700)


In [19]:

def orient_subject_matrix(X_raw, analysis_type):
    Xtv = coerce_TxV(X_raw)
    return Xtv if analysis_type == "spatial" else Xtv.T

def build_subject_objects(subject_list, analysis_type):
    return [Subject(X=orient_subject_matrix(x, analysis_type), sX=orient_subject_matrix(x, analysis_type)) for x in subject_list]

def expected_shape_string(example_X_raw, analysis_type, K):
    Xtv = coerce_TxV(example_X_raw)
    T, V = Xtv.shape
    return f"sXC ~ ({T}, {K}), S ~ ({K}, {V})" if analysis_type=="spatial" else f"sXC ~ ({V}, {K}), S ~ ({K}, {T})"

def fit_msaa(subject_list, K, analysis_type, opts=None, verbose=True):
    opts = dict(MSAA_OPTS if opts is None else opts)
    subj_objs = build_subject_objects(subject_list, analysis_type)
    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(subj_objs, noc=K, opts=opts)
    if verbose:
        print(f"[{analysis_type.upper()} AA] K={K} | VarExpl={varexpl*100:.2f}%")
        print("subj0 sXC shape:", np.asarray(results_subj[0]["sXC"]).shape)
        print("subj0 S shape:", np.asarray(results_subj[0]["S"]).shape)
        print("Expected:", expected_shape_string(subject_list[0], analysis_type, K))
    return {"results_subj": results_subj, "C": C, "cost_fun": cost_fun, "varexpl": varexpl, "secs": secs, "K": K}

def save_fit_npz(save_path, fit_res, condition_labels_str):
    np.savez_compressed(
        save_path,
        K=np.array(fit_res["K"]),
        results_subj=np.array(fit_res["results_subj"], dtype=object),
        C=np.array(fit_res["C"], dtype=object),
        cost_fun=np.array(fit_res["cost_fun"], dtype=object),
        varexpl=np.array(fit_res["varexpl"]),
        secs=np.array(fit_res["secs"]),
        condition_labels_str=np.array(condition_labels_str, dtype=object),
        condition_codes=np.array(pd.Categorical(condition_labels_str).codes),
        condition_names=np.array(sorted(np.unique(condition_labels_str)), dtype=object),
        analysis_type=np.array(ANALYSIS_TYPE, dtype=object),
        fit_scope=np.array(FIT_SCOPE, dtype=object),
        centers=np.array(centers),
        widths=np.array(widths),
    )

def get_fit_jobs(condition_data, fit_scope):
    jobs = []
    if fit_scope == "within":
        for cond_name, subject_list in condition_data.items():
            jobs.append({"job_name": cond_name, "subject_list": subject_list, "condition_labels_str": [cond_name]*len(subject_list)})
    else:
        all_subjects, all_labels = [], []
        for cond_name, subject_list in condition_data.items():
            all_subjects.extend(subject_list)
            all_labels.extend([cond_name]*len(subject_list))
        jobs.append({"job_name": "acrossCond", "subject_list": all_subjects, "condition_labels_str": all_labels})
    return jobs

def job_output_path(output_dir, analysis_type, fit_scope, job_name, K):
    return os.path.join(output_dir, f"{analysis_type}AA_{fit_scope}_{job_name}_K{K}.npz")

jobs = get_fit_jobs(condition_data, FIT_SCOPE)
jobs


[{'job_name': 'intact',
  'subject_list': [array([[ 0.7390178 ,  0.0482161 ,  0.41048738, ...,  2.08967   ,
            1.7098577 , -0.51149654],
          [ 0.99505675,  0.37382004, -1.9477133 , ...,  1.7763319 ,
            3.687591  , -0.91213423],
          [ 2.1640973 ,  0.33549806, -1.9503074 , ...,  1.2327224 ,
            2.0651553 , -0.98489267],
          ...,
          [-0.04969338,  0.3840668 , -0.60217726, ...,  0.8902922 ,
            0.5947482 , -1.0532358 ],
          [ 0.5482102 ,  1.1633033 , -1.3218285 , ...,  1.6571469 ,
           -0.40749523, -0.61102223],
          [-0.7863177 ,  0.03553534, -0.87664074, ...,  1.3443426 ,
           -1.1747104 ,  0.23313299]], dtype=float32),
   array([[-1.3355055 ,  0.05815953, -0.6389527 , ..., -2.5327556 ,
            0.05553686,  0.09448261],
          [-0.11980657, -0.90872073, -0.32386687, ..., -2.6671445 ,
           -1.0529251 ,  0.4178725 ],
          [ 0.7181586 ,  0.32694626,  1.5326638 , ..., -1.3597811 ,
           -

In [20]:

saved_paths = []
failed_jobs = []

for K in K_VALUES:
    print("\n" + "#" * 90)
    print(f"Beginning all jobs for K={K}")
    print("#" * 90)
    for job in jobs:
        job_name = job["job_name"]
        subject_list = job["subject_list"]
        condition_labels_str = job["condition_labels_str"]
        save_path = job_output_path(OUTPUT_DIR, ANALYSIS_TYPE, FIT_SCOPE, job_name, K)
        if SKIP_EXISTING and os.path.exists(save_path):
            print(f"Skipping existing: {save_path}")
            saved_paths.append(save_path)
            continue
        try:
            fit_res = fit_msaa(subject_list, K, ANALYSIS_TYPE, opts=MSAA_OPTS, verbose=VERBOSE_PROGRESS)
            save_fit_npz(save_path, fit_res, condition_labels_str)
            saved_paths.append(save_path)
            print("Saved:", save_path)
        except Exception as e:
            failed_jobs.append({"job_name": job_name, "K": K, "error_type": type(e).__name__, "error_message": str(e)})
            print(f"FAILED: job={job_name}, K={K}, error={type(e).__name__}: {e}")
            if STOP_ON_ERROR:
                raise
saved_paths, failed_jobs



##########################################################################################
Beginning all jobs for K=5
##########################################################################################
[SPATIAL AA] K=5 | VarExpl=28.06%
subj0 sXC shape: (300, 5)
subj0 S shape: (5, 700)
Expected: sXC ~ (300, 5), S ~ (5, 700)
Saved: msaa_flexible_outputs_npz/spatialAA_within_intact_K5.npz
[SPATIAL AA] K=5 | VarExpl=31.14%
subj0 sXC shape: (300, 5)
subj0 S shape: (5, 700)
Expected: sXC ~ (300, 5), S ~ (5, 700)
Saved: msaa_flexible_outputs_npz/spatialAA_within_word_K5.npz
[SPATIAL AA] K=5 | VarExpl=19.48%
subj0 sXC shape: (300, 5)
subj0 S shape: (5, 700)
Expected: sXC ~ (300, 5), S ~ (5, 700)
Saved: msaa_flexible_outputs_npz/spatialAA_within_rest_K5.npz

##########################################################################################
Beginning all jobs for K=10
##########################################################################################
Skipping existing: msa

(['msaa_flexible_outputs_npz/spatialAA_within_intact_K5.npz',
  'msaa_flexible_outputs_npz/spatialAA_within_word_K5.npz',
  'msaa_flexible_outputs_npz/spatialAA_within_rest_K5.npz',
  'msaa_flexible_outputs_npz/spatialAA_within_intact_K10.npz',
  'msaa_flexible_outputs_npz/spatialAA_within_word_K10.npz',
  'msaa_flexible_outputs_npz/spatialAA_within_rest_K10.npz'],
 [])